In [1]:
import os

# Definisci il percorso
percorso_cartella = "/kaggle/working/Semantic"

# Crea la cartella
os.makedirs(percorso_cartella, exist_ok=True)

print(f"Cartella creata in: {percorso_cartella}")

Cartella creata in: /kaggle/working/Semantic


Import dei pacchetti necessari

In [2]:
# Kaggle: spesso molte librerie sono già presenti, ma così vai sul sicuro
!pip install -q datasets transformers tokenizers evaluate rouge_score sentencepiece huggingface_hub nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00a 0:00:01


Import

In [3]:
import os
import torch
import numpy as np

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)

import evaluate


2026-01-18 17:01:50.275860: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768755710.770265      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768755710.917707      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768755712.085071      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768755712.085121      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768755712.085124      55 computation_placer.cc:177] computation placer alr

CSV ➜ JSONL

In [4]:
import os
import shutil
import json
import re
import unicodedata
import pandas as pd
from pathlib import Path

# =========================
# Percorsi (Kaggle Input)
# =========================
CSV_PATH = "/kaggle/input/nuovo-data/dataset_nuovo.csv"
JSON_IN  = "/kaggle/input/nuovo-data/dataset_nuovo.json"

# Manteniamo lo stesso nome usato nel resto del notebook
JSON_OUT   = "/kaggle/working/dataset_nuovo.json"         # JSONL grezzo
JSON_CLEAN = "/kaggle/working/dataset_nuovo_clean.jsonl"  # JSONL pulito (usa questo per load_dataset)
BAD_PATH   = "/kaggle/working/bad_rows.txt"

# =========================
# Helpers
# =========================
def looks_like_jsonl(path: str, n_lines: int = 3) -> bool:
    """Heuristics: JSONL => ogni riga dovrebbe iniziare con '{'."""
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            lines = [l.strip() for l in f.readlines()[:n_lines] if l.strip()]
        return len(lines) > 0 and all(l.startswith("{") for l in lines)
    except Exception:
        return False

def read_csv_robust(path: str) -> pd.DataFrame:
    """
    Legge un CSV delimitato da ';' con encoding robusto (per accenti e BOM).
    """
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, sep=";", encoding=enc)
        except Exception:
            pass
    # fallback autodetect
    return pd.read_csv(path, sep=None, engine="python")

def normalize_text(s: str) -> str:
    """
    Normalizza unicode (accenti), rimuove BOM e caratteri invisibili,
    converte virgolette tipografiche in virgolette standard.
    """
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\ufeff", "").strip()
    s = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", s)  # controlli invisibili
    s = (s.replace("“", '"').replace("”", '"').replace("„", '"')
           .replace("«", '"').replace("»", '"')
           .replace("’", "'").replace("‘", "'"))
    # se finisce con virgola (jsonl sporco)
    if s.endswith(","):
        s = s[:-1].rstrip()
    return s

def repair_unescaped_quotes_in_values(line: str) -> str:
    """
    Ripara casi tipo: ..."chiamare "famiglia" due uomini"...
    Escapa i doppi apici che sono dentro ai valori stringa.
    Best effort.
    """
    out = []
    in_string = False
    escape = False

    for idx, ch in enumerate(line):
        if in_string:
            if escape:
                out.append(ch)
                escape = False
            else:
                if ch == "\\":
                    out.append(ch)
                    escape = True
                elif ch == '"':
                    j = idx + 1
                    while j < len(line) and line[j] in " \t\r\n":
                        j += 1
                    # se dopo è un separatore JSON, allora chiude la stringa
                    if j < len(line) and line[j] in [",", "}", "]", ":"]:
                        out.append('"')
                        in_string = False
                    else:
                        # altrimenti è quote "interno" al testo => escapalo
                        out.append('\\"')
                else:
                    out.append(ch)
        else:
            if ch == '"':
                out.append('"')
                in_string = True
            else:
                out.append(ch)

    return "".join(out)

def clean_jsonl(in_path: str, out_path: str, bad_path: str):
    """
    Legge JSONL, prova parse + fix (quote interne), scrive JSONL pulito.
    Scarta le righe irriparabili e le logga.
    """
    good, bad = 0, 0
    bad_examples = []

    with open(in_path, "r", encoding="utf-8", errors="replace") as fin, \
         open(out_path, "w", encoding="utf-8") as fout, \
         open(bad_path, "w", encoding="utf-8") as fbad:

        for i, raw in enumerate(fin, start=1):
            line = normalize_text(raw)
            if not line or line in ["[", "]"]:
                continue

            # Tentativo 1: parse diretto
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                # Tentativo 2: riparazione quote interne
                try:
                    obj = json.loads(repair_unescaped_quotes_in_values(line))
                except json.JSONDecodeError as e:
                    bad += 1
                    fbad.write(f"RIGA {i}: {e}\n{line}\n\n")
                    if len(bad_examples) < 5:
                        bad_examples.append((i, str(e), line[:240]))
                    continue

            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            good += 1

    print(f"✅ JSON_CLEAN creato. Validi: {good} | Scartati: {bad}")
    print("JSON_CLEAN:", out_path)
    print("BAD log   :", bad_path)

    if bad_examples:
        print("\nEsempi righe scartate (max 5):")
        for i, err, snippet in bad_examples:
            print(f"- Riga {i}: {err}\n  {snippet}\n")

# =========================
# 1) Se esiste il JSON, preferiscilo
# =========================
if os.path.exists(JSON_IN):
    if looks_like_jsonl(JSON_IN):
        shutil.copyfile(JSON_IN, JSON_OUT)
        print("Trovato JSONL in input. Copiato in:", JSON_OUT)
    else:
        # Probabile JSON array -> convertiamo a JSONL
        try:
            df = pd.read_json(JSON_IN)
            df.to_json(JSON_OUT, orient="records", force_ascii=False, lines=True)
            print("JSON array convertito in JSONL:", JSON_OUT)
        except ValueError:
            # Se read_json fallisce, fallback al CSV
            print("JSON non leggibile come array: fallback al CSV")
            df = read_csv_robust(CSV_PATH)
            df.to_json(JSON_OUT, orient="records", force_ascii=False, lines=True)
            print("CSV convertito in JSONL:", JSON_OUT)

# =========================
# 2) Altrimenti convertiamo dal CSV
# =========================
elif os.path.exists(CSV_PATH):
    df = read_csv_robust(CSV_PATH)
    df.to_json(JSON_OUT, orient="records", force_ascii=False, lines=True)
    print("CSV convertito in JSONL:", JSON_OUT)

else:
    raise FileNotFoundError(
        "Non trovo i file attesi. Verifica che esistano: "
        f"{CSV_PATH} e/o {JSON_IN}"
    )

# =========================
# 3) Pulizia finale (crea JSONL valido per load_dataset)
# =========================
clean_jsonl(JSON_OUT, JSON_CLEAN, BAD_PATH)

# =========================
# Debug rapido
# =========================
print("\nJSON_OUT (grezzo):", JSON_OUT)
print("Prime 2 righe del JSON_OUT:")
with open(JSON_OUT, "r", encoding="utf-8", errors="replace") as f:
    for _ in range(2):
        line = f.readline().strip()
        if not line:
            break
        print(line)

print("\nJSON_CLEAN (usa questo):", JSON_CLEAN)
print("Prime 2 righe del JSON_CLEAN:")
with open(JSON_CLEAN, "r", encoding="utf-8") as f:
    for _ in range(2):
        line = f.readline().strip()
        if not line:
            break
        print(line)


Trovato JSONL in input. Copiato in: /kaggle/working/dataset_nuovo.json
✅ JSON_CLEAN creato. Validi: 16672 | Scartati: 1
JSON_CLEAN: /kaggle/working/dataset_nuovo_clean.jsonl
BAD log   : /kaggle/working/bad_rows.txt

Esempi righe scartate (max 5):
- Riga 8462: Expecting ',' delimiter: line 1 column 197 (char 196)
  {"non_inclusiva":"Sei troppo carina per essere un ragazzo.","inclusiva":"La bellezza fisica non è limitata a una specifica identità di genere; i ragazzi trans possono essere belli., "validazione": null}


JSON_OUT (grezzo): /kaggle/working/dataset_nuovo.json
Prime 2 righe del JSON_OUT:
{"non_inclusiva": "Se non risponde subito, probabilmente non capisce.", "inclusiva": "Il tempo di elaborazione della risposta può variare e non indica mancanza di comprensione.", "validazione": null}
{"non_inclusiva": "Non può leggere, quindi non può partecipare alla discussione.", "inclusiva": "Può partecipare utilizzando alternative alla lettura tradizionale.", "validazione": null}

JSON_CLEA

Load JSON

In [5]:
# Carico il dataset dal JSONL creato/coppiato nella cella precedente
# (Nota) load_dataset("json") funziona al meglio con JSONL (1 record per riga).

data_files = {"train": "/kaggle/working/dataset_nuovo_clean.jsonl"}

dataset = load_dataset("json", data_files=data_files)
print(dataset)
print("Columns:", dataset["train"].column_names)
print("Row0:", dataset["train"][0])


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['non_inclusiva', 'inclusiva', 'validazione'],
        num_rows: 16672
    })
})
Columns: ['non_inclusiva', 'inclusiva', 'validazione']
Row0: {'non_inclusiva': 'Se non risponde subito, probabilmente non capisce.', 'inclusiva': 'Il tempo di elaborazione della risposta può variare e non indica mancanza di comprensione.', 'validazione': None}


Modello/Tokenizer/Collator

In [6]:
# Modello IT5 (Italian T5)
# Varianti disponibili: gsarti/it5-small (piu' leggero), gsarti/it5-base (bilanciato), gsarti/it5-large (piu' pesante)
# Se sei su GPU piccola (es. T4 16GB), in caso di OOM passa a it5-small o riduci batch size.

MODEL_NAME = "gsarti/it5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Se qui si blocca, di solito e' per Internet OFF su Kaggle.
# In quel caso: aggiungi il modello come Dataset/Model in input oppure abilita Internet.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)


Device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Split + preprocess + tokenize + sanity

In [7]:
from torch.utils.data import DataLoader

SOURCE_COL = "non_inclusiva"
TARGET_COL = "inclusiva"

raw = dataset["train"]
spl = raw.train_test_split(test_size=0.1, seed=42)
splits = DatasetDict({"train": spl["train"], "validation": spl["test"]})

def _norm_text(x):
    x = "" if x is None else str(x)
    return " ".join(x.split())

# Prompt più vincolante (dataset IT "così com'è")
prefix = (
    "[INCLUSIVO] "
    "Riformula la frase in modo inclusivo e rispettoso, mantenendo il tema. "
    "Non introdurre il tema del genere se non è presente. "
    "Output: una sola frase. Testo: "
)

def preprocess_function(examples):
    inputs = [prefix + _norm_text(x) for x in examples[SOURCE_COL]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding=False)

    targets = [_norm_text(x) for x in examples[TARGET_COL]]
    # (Compatibilita') alcune versioni di transformers non supportano text_target.
    try:
        labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding=False)
    except TypeError:
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(targets, max_length=128, truncation=True, padding=False)


    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = splits.map(
    preprocess_function,
    batched=True,
    remove_columns=splits["train"].column_names
)

# filtro anti-label-vuote
def has_valid_labels(ex):
    labs = ex["labels"]
    if labs is None or len(labs) == 0:
        return False
    return any(t != tokenizer.pad_token_id for t in labs)

tokenized_datasets = tokenized_datasets.filter(has_valid_labels)

print("Train size:", len(tokenized_datasets["train"]))
print("Val size  :", len(tokenized_datasets["validation"]))
if len(tokenized_datasets["train"]) == 0:
    raise ValueError("Train vuoto dopo filter: controlla colonne SOURCE/TARGET o dati nulli.")

# Sanity batch + loss
batch = next(iter(DataLoader(tokenized_datasets["validation"], batch_size=2, collate_fn=data_collator)))
print("Non-masked label tokens per sample:", (batch["labels"] != -100).sum(dim=1).tolist())

model.eval()
batch_gpu = {k: v.to(device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch_gpu)
print("Sanity loss:", float(out.loss), "isfinite:", bool(torch.isfinite(out.loss)))


Map:   0%|          | 0/15004 [00:00<?, ? examples/s]

Map:   0%|          | 0/1668 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15004 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1668 [00:00<?, ? examples/s]

Train size: 15004
Val size  : 1668
Non-masked label tokens per sample: [24, 20]
Sanity loss: 5.272177219390869 isfinite: True


Training + ROUGE robusto + Trainer

In [8]:
import inspect

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    # compatibilita': alcune versioni passano oggetto con .predictions/.label_ids
    preds = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1]

    if isinstance(preds, (tuple, list)):
        preds = preds[0]

    preds = np.array(preds)

    # logits -> ids (capita se predict_with_generate=False)
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    preds = np.nan_to_num(
        preds,
        nan=tokenizer.pad_token_id,
        posinf=tokenizer.pad_token_id,
        neginf=tokenizer.pad_token_id,
    )
    preds = np.rint(preds).astype(np.int64)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id).astype(np.int64)

    vocab_max = int(getattr(tokenizer, "vocab_size", model.config.vocab_size)) - 1
    preds = np.clip(preds, 0, vocab_max)
    labels = np.clip(labels, 0, vocab_max)

    decoded_preds = tokenizer.batch_decode(preds.tolist(), skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)

    # per IT meglio senza stemmer
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {k: v * 100 for k, v in result.items()}

# ---- TrainingArguments: eval_strategy vs evaluation_strategy (compatibilita') ----
TA_SIG = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
_eval_key = "eval_strategy" if "eval_strategy" in TA_SIG else "evaluation_strategy"

training_args_kwargs = dict(
    output_dir="/kaggle/working/model_out_it",
    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    save_strategy="epoch",
    save_total_limit=2,

    learning_rate=5e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    logging_strategy="steps",
    logging_steps=50,
    report_to="none",

    # (Suggerimento) Se hai GPU, fp16 accelera parecchio. Se vedi instabilita', metti False.
    fp16=torch.cuda.is_available(),
    seed=42,
)

training_args_kwargs[_eval_key] = "epoch"
training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

# ---- Trainer: processing_class vs tokenizer (compatibilita') ----
TR_SIG = inspect.signature(Seq2SeqTrainer.__init__).parameters
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

if "processing_class" in TR_SIG:
    trainer = Seq2SeqTrainer(processing_class=tokenizer, **trainer_kwargs)
else:
    trainer = Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kwargs)

print("Trainer pronto. Train size:", len(tokenized_datasets["train"]), "Val size:", len(tokenized_datasets["validation"]))


Trainer pronto. Train size: 15004 Val size: 1668


Train + evaluate + sample pred/ref

In [9]:
trainer.train()
print("EVAL:", trainer.evaluate())

pred = trainer.predict(tokenized_datasets["validation"])

preds = pred.predictions
if isinstance(preds, (tuple, list)):
    preds = preds[0]

preds = np.array(preds)
if preds.ndim == 3:
    preds_ids = np.argmax(preds, axis=-1)
else:
    preds_ids = np.nan_to_num(
        preds,
        nan=tokenizer.pad_token_id,
        posinf=tokenizer.pad_token_id,
        neginf=tokenizer.pad_token_id
    )
    preds_ids = np.rint(preds_ids).astype(np.int64)

vocab_max = int(getattr(tokenizer, "vocab_size", model.config.vocab_size)) - 1
preds_ids = np.clip(preds_ids, 0, vocab_max)

labels_ids = np.where(pred.label_ids != -100, pred.label_ids, tokenizer.pad_token_id).astype(np.int64)
labels_ids = np.clip(labels_ids, 0, vocab_max)

print("\n--- SAMPLE 0 ---")
print("PRED:", tokenizer.decode(preds_ids[0].tolist(), skip_special_tokens=True))
print("REF :", tokenizer.decode(labels_ids[0].tolist(), skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.738100,1.630213,22.062691,8.766991,20.476872,20.483202
2,1.683400,1.564208,23.330393,9.964794,21.754578,21.755630
3,1.635700,1.545895,24.057834,10.120872,22.214006,22.229530


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


EVAL: {'eval_loss': 1.545894742012024, 'eval_rouge1': 24.05783424761839, 'eval_rouge2': 10.120871620599173, 'eval_rougeL': 22.21400600003996, 'eval_rougeLsum': 22.229529875377878, 'eval_runtime': 179.7067, 'eval_samples_per_second': 9.282, 'eval_steps_per_second': 1.163, 'epoch': 3.0}

--- SAMPLE 0 ---
PRED: le persone cieche hanno un sesto senso magico.
REF : le persone cieche sviluppano abilità compensative, non hanno poteri soprannaturali.


Test esempi 

In [13]:
model.eval()
model.to(device)

examples = [
    "Le donne non sanno fare altro che cucinare.",
    "I disabili sono scarsi a calcio.",
    "Le bionde sono stupide.",
    "Le donne devono solo cucinare.",
    "Le persone basse hanno difficoltà a correre a causa delle loro gambe corte.",
    "I professori di ingegneria sono meno preparati e più severi dei professori di informatica.",
]

inputs = [prefix + s for s in examples]
enc = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    gen = model.generate(
        **enc,
        max_new_tokens=64,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        length_penalty=1.1,
        do_sample=False
    )

outs = tokenizer.batch_decode(gen, skip_special_tokens=True)

for s, o in zip(examples, outs):
    print("Input (non-inclusiva):", s)
    print("Output (inclusiva):   ", o)
    print()


Input (non-inclusiva): Le donne non sanno fare altro che cucinare.
Output (inclusiva):    le capacità di cucinare variano da individuo a individuo, non in base al genere.

Input (non-inclusiva): I disabili sono scarsi a calcio.
Output (inclusiva):    le persone con disabilità possono giocare a calcio.

Input (non-inclusiva): Le bionde sono stupide.
Output (inclusiva):    il colore della pelle è una caratteristica individuale e non è legato al genere.

Input (non-inclusiva): Le donne devono solo cucinare.
Output (inclusiva):    la cucina è un'attività che può essere svolta da chiunque, indipendentemente dal genere.

Input (non-inclusiva): Le persone basse hanno difficoltà a correre a causa delle loro gambe corte.
Output (inclusiva):    le capacità fisiche variano da individuo a individuo, indipendentemente dall'altezza.

Input (non-inclusiva): I professori di ingegneria sono meno preparati e più severi dei professori di informatica.
Output (inclusiva):    le competenze e il livello di c